# Kairos: эмбеддинги текста и картинок

Рантайм: **GPU (T4 хватит)**. Результаты в `MyDrive/kairos_work/`, повторный запуск продолжает с места остановки.

Как устроено ускорение:
* распаковка медиа запускается в фоне (`7z` отдельным процессом) сразу после выбора файлов, и пока она идёт, GPU считает эмбеддинги текста;
* порции медиа идут конвейером: пока GPU считает порцию `k`, 7z уже распаковывает `k+1`;
* картинки декодируются параллельно в `DataLoader` (по процессу на ядро, JPEG сразу в уменьшенном размере), на GPU в fp16 батчами по 512;
* запись на Drive идёт в фоновом потоке и не тормозит GPU.

Пароль от архива берётся из секрета Colab `KAIROS_PW` (значок ключа слева, «Add new secret»,
включить «Notebook access»). Если секрета нет, то из переменной окружения `KAIROS_PW`, и только потом спрашивается.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, shutil, subprocess, getpass, warnings
import numpy as np, pandas as pd

ARCHIVES = '/content/drive/MyDrive/kairos_archives'
ARCH = f'{ARCHIVES}/lab1_core.7z.001'
WORK = '/content/drive/MyDrive/kairos_work'      # результаты, на Drive
DATA = '/content/kairos_data'                     # тексты и метки, локальный диск VM
MEDIA = '/content/kairos_media'                   # временная распаковка медиа
REPO = '/content/kairos-dataset'
os.makedirs(f'{WORK}/img', exist_ok=True)

_pw = None
def password():
    # Секрет Colab, потом переменная окружения, потом ручной ввод; спрашиваем один раз.
    global _pw
    if _pw is None:
        try:
            from google.colab import userdata
            _pw = userdata.get('KAIROS_PW')
        except Exception:
            _pw = os.environ.get('KAIROS_PW') or getpass.getpass('Пароль от архива: ')
    return _pw

def _z(args):
    return ['7z', *args, f'-p{password()}', '-y', '-mmt=on']

def z(*args):
    # 7z синхронно, возвращает stdout.
    r = subprocess.run(_z(args), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-2000:] or r.stdout[-2000:])
    return r.stdout

def z_start(*args):
    # 7z в фоне: распаковка идёт, пока ноутбук занят другим.
    return subprocess.Popen(_z(args), stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)

def z_wait(p):
    _, err = p.communicate()
    if p.returncode != 0:
        raise RuntimeError(err[-2000:])

subprocess.run('which 7z >/dev/null || apt-get -qq install -y p7zip-full', shell=True, check=True)

In [ ]:
# Оглавление архива и лёгкая часть релиза (тексты, метки, manifest).
# Первый раз из архива, дальше из кеша на Drive: распаковка tar занимает секунды.
TOC, LIGHT = f'{WORK}/toc.parquet', f'{WORK}/light.tar'
if os.path.exists(TOC):
    toc = pd.read_parquet(TOC)
else:
    rows, cur = [], {}
    for line in z('l', '-ba', '-slt', ARCH).splitlines():
        if line.startswith('Path = '):
            cur = {'path': line[7:]}
            rows.append(cur)
        elif line.startswith('Size = ') and cur:
            cur['size'] = int(line[7:] or 0)
    toc = pd.DataFrame(rows)
    toc.to_parquet(TOC)
PREFIX = toc.loc[toc['path'].str.endswith('labels.csv'), 'path'].iloc[0][:-len('labels.csv')]

if not os.path.isdir(DATA):
    if os.path.exists(LIGHT):
        subprocess.run(['tar', 'xf', LIGHT, '-C', '/content'], check=True)
    else:
        wanted = [PREFIX + p for p in ('labels.csv', 'split.json', 'media/manifest.jsonl', 'posts', 'authors')]
        z('x', ARCH, f'-o{DATA}', *wanted)
        subprocess.run(['tar', 'cf', LIGHT, '-C', '/content', 'kairos_data'], check=True)
DATA_DIR = os.path.join(DATA, PREFIX) if PREFIX else DATA

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Danil482/kairos-dataset.git', REPO], check=True)
sys.path.insert(0, REPO)
from kairos.data import load_posts, load_labels, load_split, load_followers

posts = load_posts(DATA_DIR)          # только свои посты, вместе с неразмеченной историей
print(f'{len(posts):,} own posts, данные в {DATA_DIR}')

## Какие медиа берём

Только свои посты. Аватарки и логотипы не берём: они постоянны у автора, а метка считается относительно автора.
Видео заменяем постером (`video_poster`, он в core). На пост и вид не больше `MAX_PER_POST` файлов.
`from_reshared_post` оставляем: у своего поста с репостом внутри это то, что видит читатель.

Ячейка сразу запускает распаковку первой порции в фоне.

In [ ]:
KINDS = ['post_image', 'document_cover', 'document_page', 'article_image', 'video_poster', 'celebration_image']
MAX_PER_POST = 4

man = pd.read_json(f'{DATA_DIR}/media/manifest.jsonl', lines=True)
m = (man[man['kind'].isin(KINDS)]
     .drop_duplicates(['author', 'post', 'kind', 'file'])
     .merge(posts[['author', 'post']], on=['author', 'post']))
m['order'] = m.groupby(['author', 'post', 'kind']).cumcount()
m = m[m['order'] < MAX_PER_POST].copy()
m['sha'] = m['file'].str.split('/').str[1].str.split('.').str[0]
m.to_parquet(f'{WORK}/post_media.parquet')

files = m.drop_duplicates('file')[['file', 'sha', 'bytes', 'kind']]
print(f"строк {len(m):,}, постов с медиа {m[['author', 'post']].drop_duplicates().shape[0]:,}, "
      f"уникальных файлов {len(files):,}, {files['bytes'].sum() / 1e9:.1f} GB")
print(files.groupby('kind')['bytes'].agg(['count', 'sum']).assign(GB=lambda x: x['sum'] / 1e9))

# Что уже посчитано в прошлых сессиях.
done = set()
for p in glob.glob(f'{WORK}/img/part_*.npz'):
    done.update(np.load(p, allow_pickle=True)['sha'])
todo = files[~files['sha'].isin(done)].sort_values('file')

# Порции по 40 % свободного места: на диске одновременно лежат две (текущая и следующая).
budget = max(1, int(shutil.disk_usage('/content').free * 0.4))
chunks = [c for _, c in todo.groupby(todo['bytes'].cumsum() // budget)]
print(f'готово {len(done):,}, осталось {len(todo):,} файлов, порций {len(chunks)}')

def start_chunk(k):
    d = f'{MEDIA}/{k}'
    os.makedirs(d, exist_ok=True)
    with open(f'{d}.txt', 'w') as f:
        f.write('\n'.join(PREFIX + 'media/' + chunks[k]['file']) + '\n')
    return z_start('x', ARCH, f'-o{d}', f'@{d}.txt')

procs = {0: start_chunk(0)} if chunks else {}

## Диагностика: 25 меток, которые не воспроизвелись

Ожидание: у всех `resid` почти ровно на пороге (`d_low` или `d_high` порядка 1e-15), то есть это шум float между версиями библиотек.
Обучение и оценка всё равно используют `labels.csv`.

In [ ]:
from kairos.label import relabel
ours = relabel(posts, load_split(DATA_DIR)['train'])
low, high = ours.attrs['cutoffs']
d = load_labels(DATA_DIR).merge(ours[['author', 'post', 'label', 'resid']],
                                on=['author', 'post'], suffixes=('', '_ours'))
d = d[d['label'] != d['label_ours']].assign(d_low=lambda x: x['resid'] - low,
                                            d_high=lambda x: x['resid'] - high)
print(pd.__version__, np.__version__)
print(d[['split', 'label', 'label_ours', 'resid', 'd_low', 'd_high']].to_string())

## Текст: multilingual-e5-base

Считается на GPU, пока 7z распаковывает медиа. Результат: `text_e5.npz` (author, post, emb float16, 768).
sentence-transformers сам сортирует тексты по длине, так что паддинга в батчах почти нет.

In [ ]:
TEXT_OUT = f'{WORK}/text_e5.npz'
if not os.path.exists(TEXT_OUT):
    subprocess.run('pip -q install sentence-transformers', shell=True, check=True)
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer('intfloat/multilingual-e5-base', device='cuda').half()
    st.max_seq_length = 512
    E = st.encode(('query: ' + posts['text']).tolist(), batch_size=256,
                  normalize_embeddings=True, show_progress_bar=True,
                  convert_to_numpy=True).astype(np.float16)
    np.savez(TEXT_OUT, author=posts['author'].to_numpy(), post=posts['post'].to_numpy(), emb=E)
    del st
    import torch; torch.cuda.empty_cache()
print(np.load(TEXT_OUT, allow_pickle=True)['emb'].shape)

## Картинки: CLIP ViT-B/32

Результат: `img/part_*.npz` (sha, emb float16 512, ok). Битые файлы получают `ok=False`.

In [ ]:
subprocess.run('pip -q install open_clip_torch', shell=True, check=True)
import torch, open_clip
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

torch.backends.cudnn.benchmark = True
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k', device='cuda')
model = model.eval().half()

class Images(torch.utils.data.Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        try:
            im = Image.open(self.paths[i])
            im.draft('RGB', (448, 448))   # JPEG декодируется сразу в уменьшенном виде
            return preprocess(im.convert('RGB')), True
        except Exception:
            return torch.zeros(3, 224, 224), False

SAVE_EVERY = 20_000

@torch.no_grad()
def embed_stream(chunk, root):
    # Отдаёт (кусок chunk, emb, ok) каждые SAVE_EVERY файлов. Эмбеддинги копятся на GPU
    # и переносятся на CPU одним куском, без синхронизации на каждом батче.
    paths = [os.path.join(root, f) for f in chunk['file']]
    loader = torch.utils.data.DataLoader(Images(paths), batch_size=512, num_workers=os.cpu_count(),
                                         pin_memory=True, prefetch_factor=4)
    buf, oks, start, n = [], [], 0, 0
    for x, ok in tqdm(loader, desc=os.path.basename(os.path.dirname(root.rstrip('/')))):
        e = model.encode_image(x.cuda(non_blocking=True).half())
        buf.append(F.normalize(e.float(), dim=-1).half())
        oks.append(ok)
        n += len(ok)
        if n - start >= SAVE_EVERY:
            yield chunk.iloc[start:n], torch.cat(buf).cpu().numpy(), torch.cat(oks).numpy()
            buf, oks, start = [], [], n
    if buf:
        yield chunk.iloc[start:n], torch.cat(buf).cpu().numpy(), torch.cat(oks).numpy()

In [ ]:
saver = ThreadPoolExecutor(1)    # запись на Drive в фоне
pending = []
for k, chunk in enumerate(chunks):
    t0 = time.time()
    z_wait(procs.pop(k))
    print(f'порция {k}: ждали распаковку {time.time() - t0:.0f}s', flush=True)
    if k + 1 < len(chunks):
        procs[k + 1] = start_chunk(k + 1)            # следующая распаковывается, пока GPU занят
    for sub, emb, ok in embed_stream(chunk, f'{MEDIA}/{k}/{PREFIX}media'):
        pending.append(saver.submit(np.savez, f'{WORK}/img/part_{time.time_ns()}.npz',
                                    sha=sub['sha'].to_numpy(), emb=emb, ok=ok))
        print(f'  {len(sub):,} файлов, битых {(~ok).sum()}', flush=True)
    shutil.rmtree(f'{MEDIA}/{k}')
for f in pending:
    f.result()
print('готово')